# 4.14 · 分位数回归 / Quantile Regression

> **课程定位 / Where this fits**
> 第 14 课，**Part 4 · 监督学习：回归**。
> Lesson 14, **Part 4 · Supervised Regression**.
>
> 前面所有回归都预测**条件均值**（"平均会是多少"）。但很多实战问题需要别的：要**预测区间**（"90% 的情况不超过多少"——库存、风控、交付时间），或目标分布**偏态/异方差**导致均值不够用。**分位数回归**直接预测**条件分位数**（中位数、P10、P90），把"波动范围"也建出来。
> All regressions so far predict the **conditional mean**. But many real problems need more: a **prediction interval** ("under what value 90% of the time" — inventory, risk, delivery time), or the target is **skewed/heteroscedastic** so the mean isn't enough. **Quantile regression** directly predicts **conditional quantiles** (median, P10, P90), modeling the spread itself.
>
> 💼 **实战/面试视角**："怎么给预测区间 / 中位数回归是什么 / pinball loss" 偏概率预测/风控/供应链岗。
> 💼 **Practical/interview angle:** "how to give prediction intervals / median regression / pinball loss" — probabilistic forecasting / risk / supply-chain roles.

> 💡 **面试相关 / Interview-relevant**
> - "分位数回归 vs 普通回归区别"（出镜率 ★★★★，分位数 vs 均值）
> - "pinball/quantile loss 是什么"（★★★★）
> - "中位数回归为什么对异常值稳健"（★★★★）
> - "怎么用分位数做预测区间"（★★★★★）

---

## 学习目标 / Learning Objectives

1. 理解 **pinball loss** 如何让模型预测某个分位数。
   Understand how the **pinball loss** makes a model predict a quantile.
2. 用线性分位数回归捕捉**异方差**。
   Capture **heteroscedasticity** with linear quantile regression.
3. 用 GBDT 分位数回归处理**非线性 + 异方差**。
   Handle nonlinearity + heteroscedasticity with GBDT quantile regression.
4. 用 P10-P90 构造**预测区间**并验证覆盖率。
   Build a P10-P90 **prediction interval** and check its coverage.

## 目录 / TOC
1. [先建直觉 + pinball loss ⭐](#1)
2. [📈 异方差数据](#2)
3. [线性分位数回归 + 预测区间 ⭐](#3)
4. [GBDT 分位数回归 ⭐](#4)
5. [覆盖率验证 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先建直觉 + pinball loss ⭐ / Intuition & the Pinball Loss

普通回归用平方误差，对称地惩罚高估和低估，所以最优解是**均值**。要预测一个**分位数**，只需把损失改成**非对称**的——对高估和低估罚不同的力度。这就是 **pinball loss（分位数损失）**：
Ordinary regression uses squared error, penalizing over- and under-estimation symmetrically, so the optimum is the **mean**. To predict a **quantile**, just make the loss **asymmetric** — penalize over- and under-estimation differently. That's the **pinball (quantile) loss**:

$$L_\tau(y, \hat y) = \begin{cases} \tau\,(y-\hat y) & y \ge \hat y \text{（低估）}\\ (\tau-1)(y-\hat y) & y < \hat y \text{（高估）}\end{cases}$$

- $\tau=0.5$：两侧对称（斜率 0.5/0.5）= **绝对误差(MAE)**，最优解是**中位数**。
  $\tau=0.5$: symmetric (slopes 0.5/0.5) = **MAE**, optimum is the **median**.
- $\tau=0.9$：低估罚得重（斜率 0.9）、高估罚得轻（0.1）→ 把模型"往高里推"，最优解是 **90 分位**。
  $\tau=0.9$: heavy penalty for under-estimation (slope 0.9), light for over (0.1) → pushes predictions up, optimum is the **90th percentile**.

直觉：罚谁罚得狠，模型就尽量避开谁，从而落在对应的分位数上。
Intuition: whichever side is penalized harder, the model avoids it, landing on the corresponding quantile.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

err = np.linspace(-3, 3, 300)        # err = y - ŷ; >0 表示低估(模型预测偏低)
fig, ax = plt.subplots(figsize=(7, 4))
for tau, c in [(0.1, "C0"), (0.5, "C1"), (0.9, "C2")]:
    loss = np.where(err >= 0, tau*err, (tau-1)*err)   # pinball: 两侧不同斜率
    ax.plot(err, loss, color=c, lw=2, label=f"τ={tau}")
ax.axvline(0, color="k", lw=0.5); ax.legend()
ax.set_xlabel("误差 error (y - ŷ)"); ax.set_ylabel("pinball loss")
ax.set_title("Pinball loss: 非对称 V 形\nτ=0.9 时低估(右)惩罚陡, τ=0.5 对称=MAE")
plt.tight_layout(); plt.show()
print("τ=0.9: 右侧(低估)斜率陡(0.9), 左侧(高估)平(0.1) → 推模型往高预测 = 高分位")
print("τ=0.5: 两侧对称 = 绝对误差 MAE → 最优解是中位数(故中位数回归对异常值稳健)")


<a id="2"></a>
## 2. 异方差数据 / Heteroscedastic Data

分位数回归最闪光的场景是**异方差**——目标的波动随特征变化（4.2 见过的"喇叭形"）。造一份噪声标准差随 x 线性增大的数据：x 小时点很集中，x 大时散得很开。单一均值线只能给中心趋势，**说不出"波动越来越大"**。
Quantile regression shines under **heteroscedasticity** — the target's spread varies with features (the "fan shape" from 4.2). We make data whose noise std grows linearly with x: tight for small x, wide for large x. A single mean line gives only the central trend and **can't express "growing spread"**.


In [ ]:
n = 1000
x = np.sort(rng.uniform(0, 10, n))
y = 2*x + rng.normal(0, 0.5 + 0.6*x, n)    # 噪声 std = 0.5+0.6x, 随 x 线性增大!
X = x.reshape(-1, 1)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, alpha=0.2, s=10)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("异方差数据: 散布随 x 张开(喇叭形)\n均值线无法表达'波动越来越大'")
plt.tight_layout(); plt.show()
print("数据呈喇叭形(4.2 异方差): x 小时点集中, x 大时散开")
print("单一均值线只给中心趋势; 分位数回归能描述'张开的宽度'")


<a id="3"></a>
## 3. 线性分位数回归 + 预测区间 ⭐ / Linear QR & Prediction Intervals

`QuantileRegressor` 用 pinball loss 拟合指定分位数。同时拟合 P10、P50、P90 三条线：你会看到它们**随 x 张开**（捕捉了异方差），而 OLS 只有一条均值线。更实用的是：**P10 和 P90 之间就是一个 80% 的预测区间**——这比单点预测信息丰富得多（"预测值是 X，且 80% 概率落在 [a, b]"）。
`QuantileRegressor` fits a given quantile with the pinball loss. Fitting P10, P50, P90 together, you see them **fan out with x** (capturing heteroscedasticity), while OLS gives one mean line. More usefully: **the band between P10 and P90 is an 80% prediction interval** — far more informative than a point prediction.


In [ ]:
from sklearn.linear_model import QuantileRegressor, LinearRegression

x_plot = np.linspace(0, 10, 100).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.15, s=8)
ax.plot(x_plot, LinearRegression().fit(X, y).predict(x_plot), "k--", lw=2, label="OLS 均值 mean")

preds = {}
for q, c in [(0.1, "C0"), (0.5, "C1"), (0.9, "C2")]:
    # quantile=q 指定分位数; alpha=0 表示不加正则; solver='highs' 是线性规划求解器
    qr = QuantileRegressor(quantile=q, alpha=0, solver="highs").fit(X, y)
    preds[q] = qr.predict(x_plot)
    ax.plot(x_plot, preds[q], color=c, lw=2, label=f"分位数 quantile τ={q}")
# P10~P90 之间填充 = 80% 预测区间 / shade the prediction interval
ax.fill_between(x_plot.ravel(), preds[0.1], preds[0.9], alpha=0.12, color="green", label="80% 预测区间(P10-P90)")
ax.legend(fontsize=9); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("分位数回归: 分位数线张开=捕捉异方差; P10-P90=80% 预测区间")
plt.tight_layout(); plt.show()
print("三条分位数线随 x 张开(OLS 只有一条均值线) → 直接显示波动随 x 增大")
print("P10 到 P90 之间 = 80% 预测区间, 比单点预测信息丰富得多")


<a id="4"></a>
## 4. GBDT 分位数回归 ⭐ / GBDT Quantile Regression

分位数回归不限于线性。GBDT(4.13) 只要把损失换成 quantile loss，就能同时捕捉**非线性 + 异方差**——这正是 4.13 "boosting 换损失即可适配不同任务" 的体现。`GradientBoostingRegressor(loss="quantile", alpha=τ)` 即可。
Quantile regression isn't limited to linear models. GBDT (4.13) only needs the quantile loss to capture **nonlinearity + heteroscedasticity** together — exactly the "swap the loss to adapt boosting" idea from 4.13. Use `GradientBoostingRegressor(loss="quantile", alpha=τ)`.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

y_nl = np.sin(x) * x + rng.normal(0, 0.3 + 0.5*x, n)   # 非线性曲线 + 异方差噪声
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y_nl, alpha=0.15, s=8)
gp = {}
for q, c in [(0.1,"C0"),(0.5,"C1"),(0.9,"C2")]:
    # loss='quantile' + alpha=q 让 GBDT 预测第 q 分位 / GBDT with quantile loss
    g = GradientBoostingRegressor(loss="quantile", alpha=q, n_estimators=100,
                                  max_depth=3, learning_rate=0.1, random_state=0).fit(X, y_nl)
    gp[q] = g.predict(x_plot)
    ax.plot(x_plot, gp[q], color=c, lw=2, label=f"GBDT τ={q}")
ax.fill_between(x_plot.ravel(), gp[0.1], gp[0.9], alpha=0.12, color="green")
ax.legend(); ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("GBDT 分位数回归: 非线性 + 异方差 都能捕捉")
plt.tight_layout(); plt.show()
print("GBDT 分位数同时捕捉非线性曲线 + 张开的区间 — boosting 换损失的威力(4.13)")
print("💡 也可用 XGBoost/LightGBM 的 quantile objective, 或 quantile-forest 扩展")


<a id="5"></a>
## 5. 覆盖率验证 + 小结 ⭐ / Coverage Check & Summary

预测区间好不好，要看**覆盖率**：名义上的 80% 区间，实际是否真有约 80% 的测试点落在里面（接 2.5 置信区间的覆盖率思想）。同时用 **mean_pinball_loss** 确认每个分位数模型确实在最小化对应的损失。
A prediction interval is judged by **coverage**: does a nominal 80% interval actually contain ~80% of test points (the coverage idea from 2.5)? We also check **mean_pinball_loss** to confirm each quantile model minimizes its loss.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_pinball_loss

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
lo = QuantileRegressor(quantile=0.1, alpha=0, solver="highs").fit(X_tr, y_tr).predict(X_te)
hi = QuantileRegressor(quantile=0.9, alpha=0, solver="highs").fit(X_tr, y_tr).predict(X_te)

# 覆盖率: 实际落在 [P10,P90] 内的测试点比例, 应≈80% / empirical coverage
coverage = np.mean((y_te >= lo) & (y_te <= hi))
print(f"P10-P90 名义 80% 预测区间: 实际覆盖率 = {coverage:.1%} (目标 80%)")
print("→ 接近 80% = 分位数预测校准良好(2.5 覆盖率验证)\n")

# 每个分位数的 pinball loss, 验证模型在最小化它 / per-quantile pinball loss
for q in [0.1, 0.5, 0.9]:
    qr = QuantileRegressor(quantile=q, alpha=0, solver="highs").fit(X_tr, y_tr)
    print(f"  τ={q}: test pinball loss = {mean_pinball_loss(y_te, qr.predict(X_te), alpha=q):.3f}")


```
分位数回归: 预测条件分位数(中位数/P10/P90), 而非均值
pinball loss: 非对称 V 形; τ=0.5 对称=MAE→中位数(抗异常); τ=0.9 重罚低估→高分位
异方差: 多条分位数线随 x 张开, 捕捉"波动随特征变化"(单均值线做不到)
预测区间: P10~P90 = 80% 区间; 用覆盖率验证(实际≈名义)
不限线性: GBDT loss='quantile' 同时抓非线性+异方差(boosting 换损失 4.13)
```

### 💡 面试速查 / Interview cheat-sheet
1. **分位数回归预测分位数, 普通回归预测均值**。
   Quantile regression predicts quantiles; ordinary regression predicts the mean.
2. **pinball loss 非对称**: 罚谁狠就避开谁 → 落在对应分位数。
   Pinball loss is asymmetric: avoid the harder-penalized side → land on that quantile.
3. **中位数回归(τ=0.5)=MAE, 对异常值稳健**。
   Median regression (τ=0.5) = MAE, robust to outliers.
4. **P10-P90 = 80% 预测区间**; 用覆盖率验证校准。
   P10-P90 = 80% prediction interval; validate with coverage.
5. **GBDT/XGBoost 换 quantile loss** 即可做非线性分位数。
   Swap in quantile loss for GBDT/XGBoost to get nonlinear quantiles.

### 下一节 / Next
**4.15 稳健回归**——普通 OLS 对异常值极敏感(平方放大大误差)。稳健回归(Huber/RANSAC/Theil-Sen)在有离群点时仍能拟合出可靠的趋势。
**4.15 Robust Regression** — OLS is very sensitive to outliers (squares amplify large errors). Robust regression (Huber/RANSAC/Theil-Sen) fits a reliable trend despite outliers.
